# NTX Q1 Stage 17 — Human-Written Multilingual End-to-End Authorization

Builds a genuinely human-written, independently annotated benchmark. The notebook refuses to silently replace missing human data with synthetic text. Once adjudicated labels are provided, it computes agreement and evaluates the selected NiyamTrace runtime adapter.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

## Required input

Place `human_requests.csv` in the stage input directory with columns:

`case_id, raw_text, language`

Each request must be written by a human participant rather than generated by this notebook.

After blind annotation, place `annotations.csv` with at least two annotators per case using the supplied schema. The notebook will not label the data automatically.

In [ ]:
import pandas as pd, numpy as np
STAGE="17_human_e2e"
OUT=ensure_dir(RESULTS_ROOT/STAGE)
INP=ensure_dir(ROOT/"inputs"/STAGE)
requests_path=INP/"human_requests.csv"
annotations_path=INP/"annotations.csv"

schema_cols = ["case_id","raw_text","language","annotator_id","expected_verdict","action","entity_id","month","year","amount","currency","scope","negated","requires_clarification","notes"]

if not requests_path.exists():
    pd.DataFrame(columns=["case_id","raw_text","language"]).to_csv(INP/"human_requests_TEMPLATE.csv",index=False)
    (OUT/"HUMAN_DATA_REQUIRED.md").write_text(
        "No human_requests.csv was supplied. No synthetic substitute was generated. "
        "Collect human-written requests, then rerun this notebook.", encoding="utf-8")
    record_status(OUT,STAGE,"WAITING_FOR_HUMAN_DATA")
    print("Waiting for:", requests_path)
else:
    req=pd.read_csv(requests_path)
    need={"case_id","raw_text","language"}
    assert need.issubset(req.columns), f"Missing: {need-set(req.columns)}"
    assert req.case_id.is_unique
    # randomized blind forms
    for k,seed in [("A",SEED),("B",SEED+1),("C",SEED+2)]:
        x=req.sample(frac=1,random_state=seed).copy()
        x["annotator_id"]=k
        for c in schema_cols:
            if c not in x.columns: x[c]=""
        x[schema_cols].to_csv(OUT/f"blind_annotation_{k}.csv",index=False)
    print("Generated blind annotation forms:", len(req), "cases")

In [ ]:
# Agreement + adjudication gate.
if annotations_path.exists():
    import krippendorff
    ann=pd.read_csv(annotations_path)
    required={"case_id","annotator_id","expected_verdict","requires_clarification"}
    assert required.issubset(ann.columns), f"Missing annotation columns: {required-set(ann.columns)}"
    ann["expected_verdict"]=ann["expected_verdict"].map(normalize_verdict)
    assert ann.expected_verdict.isin(["ALLOW","BLOCK","CLARIFY"]).all()
    n_annotators=ann.annotator_id.nunique()
    counts=ann.groupby("case_id").annotator_id.nunique()
    assert counts.min()>=2, "Every case needs >=2 independent annotations."

    # Nominal Krippendorff alpha
    labels={"ALLOW":0,"BLOCK":1,"CLARIFY":2}
    wide=ann.pivot(index="annotator_id",columns="case_id",values="expected_verdict")
    rel=np.array([[labels.get(x,np.nan) for x in row] for row in wide.to_numpy()],dtype=float)
    alpha=float(krippendorff.alpha(reliability_data=rel,level_of_measurement="nominal"))

    vote=(ann.groupby("case_id")["expected_verdict"]
          .agg(lambda s: s.value_counts().index[0]).rename("expected_verdict"))
    conflicts=(ann.groupby("case_id")["expected_verdict"].nunique()>1)
    adjudicated=ann.drop_duplicates("case_id")[["case_id","raw_text","language"]].merge(vote,on="case_id")
    adjudicated["had_disagreement"]=adjudicated.case_id.map(conflicts)
    adjudicated.to_csv(OUT/"adjudicated_majority_labels.csv",index=False)
    write_json(OUT/"agreement.json",{
        "n_cases":int(adjudicated.shape[0]),
        "n_annotators":int(n_annotators),
        "krippendorff_alpha_verdict":alpha,
        "cases_with_disagreement":int(conflicts.sum())
    })
    print("Krippendorff alpha:",alpha)
else:
    print("annotations.csv not present yet; evaluation is intentionally paused.")

## Runtime evaluation

For publication, set `NTX_HUMAN_EVAL_PREDICTIONS` to a CSV produced by the actual frozen NiyamTrace runtime or current integrated pipeline. Required columns: `case_id,actual_verdict,latency_ms`.

This separation avoids inventing model/runtime predictions inside an annotation notebook.

In [ ]:
pred_path=os.environ.get("NTX_HUMAN_EVAL_PREDICTIONS","")
adj_path=OUT/"adjudicated_majority_labels.csv"
if pred_path and Path(pred_path).exists() and adj_path.exists():
    pred=pd.read_csv(pred_path)
    adj=pd.read_csv(adj_path)
    merged=adj.merge(pred,on="case_id",how="inner",validate="one_to_one")
    assert len(merged)>=min(300,len(adj)), "Too many missing runtime predictions."
    metrics=classification_metrics(merged)
    if "latency_ms" in merged:
        metrics["latency_median_ms"]=float(merged.latency_ms.median())
        metrics["latency_p95_ms"]=float(merged.latency_ms.quantile(.95))
    merged.to_csv(OUT/"human_e2e_case_results.csv",index=False)
    write_json(OUT/"human_e2e_metrics.json",metrics)
    agreement=json.loads((OUT/"agreement.json").read_text())
    paper_ready=(len(merged)>=300 and agreement["n_annotators"]>=2 and agreement["krippendorff_alpha_verdict"]>=0.67)
    record_status(OUT,STAGE,"PASS" if paper_ready else "PARTIAL",paper_ready=paper_ready,metrics=metrics,agreement=agreement)
    display(pd.DataFrame([metrics]))
else:
    print("No runtime prediction file supplied. Annotation outputs remain valid but no E2E claim is made.")

In [ ]:
z=package_stage(OUT,"NTX_Q1_17_HUMAN_E2E_RESULTS.zip")
print("Created:",z)
maybe_colab_download(z)